In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("roberta-base", padding="max_length", truncation=True)
factkb = AutoModelForSequenceClassification.from_pretrained("bunsenfeng/FactKB", num_labels = 2)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: bunsenfeng/FactKB
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = factkb.to(device)

In [ ]:
article = """
The club, which is owned by its fans via the Exeter City Supporters Trust, declared a total profit of £1.642m for 2014-15, mainly due to the sale of midfielder Matt Grimes to Swansea City.
The deal in January 2015 for £1.75m, was a record sale for the club.
The profits came in the same year that the club were forced to get an emergency loan to pay players' wages.
In June 2014 cashflow problems meant the Professional Footballers' Association stepped in to pay the players and forced Exeter into a transfer embargo which was only lifted in August of that year.
The club's accounts also show that Exeter had a surplus of shareholders' funds of £761,000, compared to a deficit of £880,000 the previous year and reduced their cost base by almost 5%.
Exeter are hoping to redevelop St James Park after being given planning permission last month.
"The board have agreed in broad terms how the income from the transfer should be spent to improve the infrastructure strength of the club in line with its vision and underlying model and these plans are now being developed in detail for further presentation," said a club statement.
"""

summary = """Exeter City have announced record profits for the last financial year."""

input = [[summary, article]]

tokens = tokenizer(input, return_tensors="pt", padding="max_length", truncation=True)
# Move tokens to the same device as the model
tokens = {k: v.to(device) for k, v in tokens.items()}
result = torch.softmax(factkb(**tokens).logits, dim = 1)

result[0]

tensor([0.6182, 0.3818], device='cuda:0', grad_fn=<SelectBackward0>)

In [ ]:
result[0]

tensor([0.9835, 0.0165], grad_fn=<SelectBackward0>)

## Batch Processing

In [ ]:
def chunk_article(
    article,
    tokenizer,
    max_length=512,
    stride=256
):
    """
    Returns a list of article chunks decoded back to text.
    """
    article_tokens = tokenizer.encode(
        article,
        add_special_tokens=False
    )

    chunks = []
    for start in range(0, len(article_tokens), stride):
        end = start + max_length
        chunk_tokens = article_tokens[start:end]

        if len(chunk_tokens) == 0:
            continue

        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)

        if end >= len(article_tokens):
            break

    return chunks

In [ ]:
def build_pairs(summary, article, tokenizer):
    chunks = chunk_article(article, tokenizer)

    pairs = [(summary, chunk) for chunk in chunks]
    return pairs

In [ ]:
def factkb_score_summary(
    summary,
    article,
    model,
    tokenizer,
    device,
    batch_size=8,
    max_length=512
):
    pairs = build_pairs(summary, article, tokenizer)

    if len(pairs) == 0:
        return 0.0, []

    all_scores = []

    model.eval()
    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch_pairs = pairs[i:i + batch_size]
            summaries, chunks = zip(*batch_pairs)

            tokens = tokenizer(
                list(summaries),
                list(chunks),
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            tokens = {k: v.to(device) for k, v in tokens.items()}
            outputs = model(**tokens)

            probs = torch.softmax(outputs.logits, dim=1)
            factual_probs = probs[:, 1]  # label=1

            all_scores.extend(factual_probs.cpu().tolist())

    # Aggregation
    final_score = max(all_scores)  # ← recommended

    return final_score, all_scores

In [ ]:
def top_k_mean(scores, k=3):
    if len(scores) == 0:
        return 0.0
    k = min(k, len(scores))
    top_k = sorted(scores, reverse=True)[:k]
    return sum(top_k) / k

In [ ]:
import torch

def factkb_score_summary(
    summary,
    article,
    model,
    tokenizer,
    device,
    batch_size=8,
    max_length=512,
    top_k=3
):
    # ---- Chunk article ----
    article_tokens = tokenizer.encode(article, add_special_tokens=False)

    chunks = []
    stride = max_length // 2

    for start in range(0, len(article_tokens), stride):
        end = start + max_length
        chunk_tokens = article_tokens[start:end]

        if not chunk_tokens:
            continue

        chunks.append(tokenizer.decode(chunk_tokens))

        if end >= len(article_tokens):
            break

    if len(chunks) == 0:
        return 0.0, []

    # ---- Inference ----
    all_scores = []
    model.eval()

    with torch.no_grad():
        for i in range(0, len(chunks), batch_size):
            batch_chunks = chunks[i:i + batch_size]

            tokens = tokenizer(
                [summary] * len(batch_chunks),
                batch_chunks,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            tokens = {k: v.to(device) for k, v in tokens.items()}
            outputs = model(**tokens)

            probs = torch.softmax(outputs.logits, dim=1)
            factual_probs = probs[:, 1]   # label=1 = factual

            all_scores.extend(factual_probs.cpu().tolist())

    # ---- Top-K Mean aggregation ----
    final_score = top_k_mean(all_scores, k=top_k)

    return final_score, all_scores

In [ ]:
final_score, chunk_scores = factkb_score_summary(
    summary=summary,
    article=article,
    model=model,
    tokenizer=tokenizer,
    device=device
)

binary_label = int(final_score >= 0.3)

print("Final factuality score:", final_score)
print("Binary label:", binary_label)
print("Per-chunk scores:", chunk_scores)

Final factuality score: 0.38181400299072266
Binary label: 1
Per-chunk scores: [0.38181400299072266]


In [ ]:
import pandas as pd

data_df = pd.read_csv("/content/drive/MyDrive/research/gold_data/expanded_triplet_evaluations_annotated_is_supported_fixed.csv")

data_df.shape

(623, 24)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
factual_label = []
factual_score = []

In [ ]:
len(data_df)

623

In [ ]:
from tqdm.auto import tqdm
import time
import torch
import json

for idx, row in tqdm(data_df.iterrows(), total=len(data_df)):
    article = row['document']
    summary = row['gold_summary']
    doc_id = row['doc_id']

    final_score, chunk_scores = factkb_score_summary(
    summary=summary,
    article=article,
    model=model,
    tokenizer=tokenizer,
    device=device
  )

    binary_label = int(final_score >= 0.3)

    factual_label.append(binary_label)
    factual_score.append(final_score)

    print(f'Doc ID: {doc_id}')
    print("Final factuality score:", final_score)
    print("Binary label: ", binary_label)

    # # Ensure GPU finishes work
    # torch.cuda.synchronize()

    # # Optional: free cached memory
    # torch.cuda.empty_cache()

    # Rate limiting / pacing (not model stress)
    # time.sleep(1)

    # if idx % 100 == 0:
    #   with open(f"fact_eval_res_{idx}.json", "w") as f:
    #     json.dump(fact_eval_res, f)

  0%|          | 0/623 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (746 > 512). Running this sequence through the model will result in indexing errors


Doc ID: 38264402
Final factuality score: 0.9986805319786072
Binary label:  1
Doc ID: 38264402
Final factuality score: 0.9986805319786072
Binary label:  1
Doc ID: 38264402
Final factuality score: 0.9986805319786072
Binary label:  1
Doc ID: 38264402
Final factuality score: 0.9986805319786072
Binary label:  1
Doc ID: 38264402
Final factuality score: 0.9986805319786072
Binary label:  1
Doc ID: 34227252
Final factuality score: 0.32956162095069885
Binary label:  1
Doc ID: 34227252
Final factuality score: 0.32956162095069885
Binary label:  1
Doc ID: 34227252
Final factuality score: 0.32956162095069885
Binary label:  1
Doc ID: 34227252
Final factuality score: 0.32956162095069885
Binary label:  1
Doc ID: 34227252
Final factuality score: 0.32956162095069885
Binary label:  1
Doc ID: 34227252
Final factuality score: 0.32956162095069885
Binary label:  1
Doc ID: 34227252
Final factuality score: 0.32956162095069885
Binary label:  1
Doc ID: 38537698
Final factuality score: 0.005114949308335781
Binary 

In [ ]:
print(len(factual_label))

623


In [ ]:
data_df['FactKB_label'] = factual_label
data_df['FactKB_score'] = factual_score

data_df.head()

,doc_id,document,gold_summary,generated_summary,document_triplets,gen_summary_triplets,summary_triplet_full,is_supported,Your_annotation_is_supported,supporting_evidence,...,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,FactKB_label,FactKB_score
0,38264402,"Prison Link Cymru had 1,099 referrals in 2015-...","There is a ""chronic"" need for more housing for...","More than 1,000 homeless people were referred ...","[['Prison Link Cymru', 'had', '1,099 referrals...","[['More than 1,000 homeless people', 'referred...","['More than 1,000 homeless people', 'referred ...",True,True,"[['Prison Link Cymru', 'had', '1,099 referrals...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0.998681
1,38264402,"Prison Link Cymru had 1,099 referrals in 2015-...","There is a ""chronic"" need for more housing for...","More than 1,000 homeless people were referred ...","[['Prison Link Cymru', 'had', '1,099 referrals...","[['More than 1,000 homeless people', 'referred...","['charity', 'located in', 'Wales']",True,True,"[['Prison Link Cymru', 'operates in', '2015-16...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0.998681
2,38264402,"Prison Link Cymru had 1,099 referrals in 2015-...","There is a ""chronic"" need for more housing for...","More than 1,000 homeless people were referred ...","[['Prison Link Cymru', 'had', '1,099 referrals...","[['More than 1,000 homeless people', 'referred...","['referral', 'occurred in', 'last year']",True,False,"[['Prison Link Cymru', 'operates in', '2015-16...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0.998681
3,38264402,"Prison Link Cymru had 1,099 referrals in 2015-...","There is a ""chronic"" need for more housing for...","More than 1,000 homeless people were referred ...","[['Prison Link Cymru', 'had', '1,099 referrals...","[['More than 1,000 homeless people', 'referred...","['More than 1,000 homeless people', 'associate...",True,True,"[['Prison Link Cymru', 'had', '1,099 referrals...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0.998681
4,38264402,"Prison Link Cymru had 1,099 referrals in 2015-...","There is a ""chronic"" need for more housing for...","More than 1,000 homeless people were referred ...","[['Prison Link Cymru', 'had', '1,099 referrals...","[['More than 1,000 homeless people', 'referred...","['charity', 'associated with', 'last year']",True,False,"[['Prison Link Cymru', 'operates in', '2015-16...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0.998681


In [ ]:
data_df['FactKB_score'] = factual_score
data_df.head()

In [ ]:
data_df.to_csv("/content/drive/MyDrive/research/factKB/fact_kb_eval_07_04_26.csv", index=False)